# `lsolve_J0`: resolution study
`N016`…`N128`: $N^3$, $L=2\pi$, $\delta t=5\times10^{-3}$, $t\le1$, $\sigma=11.9$, $\nu=5\times10^{-3}$. Rerun: `./run_all.sh [N032 …]`.

### Equations → code (`special/disp_current.f90`)
| equation | code |
|---|---|
| $\partial_t\mathbf A=-\mathbf E$ | `df(iax:iaz) -= p%el` |
| $\partial_t\mathbf E=\nabla\times\mathbf B-\mathbf J$ | `dEdt = c_light2*(p%curlb - mu0*p%jj_ohm)` |
| $\nabla\times\mathbf B=-\nabla^2\mathbf A+\nabla\Gamma$ | `p%curlb = -p%del2a + p%gGamma` |
| $\partial_t\Gamma=-(1-w)\,\nabla\cdot\mathbf E-w\,J^0$ | `df(iGamma)`, $w=$ `weight_longitudinalE` $=2$ |
| $\partial_t J^0=-\nabla\cdot\mathbf J$ | `df(irhoe) -= p%divJ` |
| $f^\mu=(\mathbf E\cdot\mathbf J,\;J^0\mathbf E+\mathbf J\times\mathbf B)$ | `p%ext_force(:,1:4)` → hydro/density (relativistic prefactors) |

### Definitions
| symbol | code | definition |
|---|---|---|
| $\mathbf E$ | `f(:,:,:,iex:iez)`, `p%el` | evolved |
| $J^0$ | `f(:,:,:,irhoe)`, `p%J0` | evolved; Gauss: $\nabla\cdot\mathbf E=J^0$ |
| $\Gamma$ | `f(:,:,:,iGamma)` | $\Gamma(0)=\nabla\cdot\mathbf A$ |
| $\gamma$ | `p%lorentz_gamma` | $(1-u^2)^{-1/2}$ |
| $\sigma$ | `sigma_J0` | $1/(\mu_0\eta)$, $\eta$ = `eta` (magnetic) |
| $\rho_e$ | `p%rhoe` | $J^0/\gamma-\sigma\,\mathbf u\cdot\mathbf E$ |
| $\mathbf W$ | `Wv` | $\mathbf E+\mathbf u\times\mathbf B-(\mathbf u\cdot\mathbf E)\,\mathbf u$ |
| $\mathbf J$ | `p%jj_ohm` | $J^0\mathbf u+\gamma\sigma\mathbf W$ |
| $r$ | — | $\nabla\cdot\mathbf E-J^0$ |

### $\nabla\cdot\mathbf J$ (`p%divJ`)
$$\nabla\cdot\mathbf J=\underbrace{\mathbf u\cdot\nabla J^0+J^0\,\nabla\cdot\mathbf u}_{\texttt{divJadv}}
+\underbrace{\sigma\gamma\,\nabla\cdot\mathbf W}_{\texttt{divJohm}}+\underbrace{\sigma\,\mathbf W\cdot\nabla\gamma}_{\texttt{divJgam}}$$
$$\nabla\cdot\mathbf W=J^0+\nabla\cdot(\mathbf u\times\mathbf B)-\mathbf u\cdot\nabla(\mathbf u\cdot\mathbf E)-(\mathbf u\cdot\mathbf E)\,\nabla\cdot\mathbf u,
\qquad \partial_i\gamma=\gamma^3u_j\,\partial_iu_j$$
| term | code |
|---|---|
| $\nabla J^0$ | `gJ0` = `grad(f,irhoe)` |
| $\nabla\cdot(\mathbf u\times\mathbf B)=\epsilon_{ijk}\partial_i(u_jB_k)$ | `tmp` (Levi-Civita loop with `p%uij`, `p%bij`) |
| $\partial_i(\mathbf u\cdot\mathbf E)=E_j\partial_iu_j+u_j\partial_iE_j$ | `guE`, with `Eij` = `gij(f,iex,Eij,1)`, $\texttt{Eij(:,j,i)}=\partial_iE_j$ |
| $\nabla\gamma$ | `ggam` |

$\nabla\cdot\mathbf E\to J^0$ in $\nabla\cdot\mathbf W$ $\;\Rightarrow\;\partial_t r\supset-\gamma\sigma\,r$ (`gaussdamprms`).

### Initial conditions (`start.in`)
| field | code | |
|---|---|---|
| $\mathbf u$ | `inituu='ABC'`, `ampluu=0.1` | $0.1(\sin z+\cos y,\ \sin x+\cos z,\ \sin y+\cos x)$ |
| $\mathbf A$ | `initaa='Beltrami-x'`, `amplaa=0.1` | $\mathbf B=\nabla\times\mathbf A$, $B_{\rm rms}=0.1$ |
| $\mathbf E$ | `initee='coswave-phase'` | $0.05\,(\cos x,\ \cos 2z,\ 0)$ |
| $J^0,\ \Gamma$ | `init_special` | $\nabla\cdot\mathbf E,\ \nabla\cdot\mathbf A$ (ghosts via `update_ghosts_ptr`) |

### Diagnostics (`print.in` → `data/time_series.dat`)
| name | quantity |
|---|---|
| `gaussJ0rms`, `gaussJ0max` | rms $r$, $\max\vert r\vert $ |
| `gaussdamprms` | rms $\gamma\sigma r$ |
| `grms` | rms$(\Gamma-\nabla\cdot\mathbf A)$ |
| `divJrms`, `divJadvrms`, `divJohmrms`, `divJgamrms` | rms $\nabla\cdot\mathbf J$ and its three terms |
| `Johmrms`, `J0urms`, `Jsigrms` | rms $\vert \mathbf J\vert $, $\vert J^0\mathbf u\vert $, $\vert \gamma\sigma\mathbf W\vert $ |
| `f0m`, `f0rms` | $\langle\mathbf E\cdot\mathbf J\rangle$, rms $\mathbf E\cdot\mathbf J$ |
| `fLorrms`, `fJ0Erms`, `fJxBrms` | rms $\vert J^0\mathbf E+\mathbf J\times\mathbf B\vert $, $\vert J^0\mathbf E\vert $, $\vert \mathbf J\times\mathbf B\vert $ |
| `uEm`, `uErms` | $\langle\mathbf u\cdot\mathbf E\rangle$, rms |
| `gammam`, `gammamax` | $\langle\gamma\rangle$, $\max\gamma$ |
| `J0rms`, `rhoerms`, `divErms` | rms $J^0$, $\rho_e$, $\nabla\cdot\mathbf E$ |

In [ ]:
import sys, re
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, LogLocator, NullLocator

PENCIL_PY = Path.home() / "pencil-code" / "python"
sys.path.insert(0, str(PENCIL_PY))
import pencil as pc

STUDY = Path.cwd() if (Path.cwd() / "run_all.sh").exists() else \
    Path.home() / "projects/Constrain_cleaning/runs/pencil/res_study"
RUNS = {16: "N016", 32: "N032", 64: "N064", 128: "N128"}
# only finished runs (Pencil writes COMPLETED at the end of run.x)
unfinished = [d for d in RUNS.values() if not (STUDY / d / "COMPLETED").exists()]
RUNS = {N: STUDY / d for N, d in RUNS.items() if (STUDY / d / "COMPLETED").exists()}
if unfinished:
    print("skipping unfinished runs:", unfinished)
NS = sorted(RUNS)
print("study dir:", STUDY)
print("runs with output:", NS)

### Plot style

In [ ]:
mpl.rcParams.update({
    "figure.figsize": (9, 6), "figure.dpi": 110,
    "font.size": 16, "axes.titlesize": 17, "axes.labelsize": 17,
    "xtick.labelsize": 15, "ytick.labelsize": 15, "legend.fontsize": 14,
    "lines.linewidth": 2.8, "lines.markersize": 9,
    "axes.linewidth": 1.5, "axes.grid": True, "grid.alpha": 0.3,
    "xtick.major.size": 7, "ytick.major.size": 7, "xtick.major.width": 1.4, "ytick.major.width": 1.4,
    "xtick.minor.size": 3.5, "ytick.minor.size": 3.5,
    "legend.frameon": True, "legend.framealpha": 0.85,
})
# one colour per resolution, ordered (light -> dark with increasing N)
COL = dict(zip([16, 32, 64, 128], plt.cm.viridis([0.85, 0.6, 0.35, 0.05])))

def style(fig):
    '''At most 5 major ticks per axis (linear or log), applied to every axis of the figure.'''
    for ax in fig.axes:
        for axis, scale in [(ax.xaxis, ax.get_xscale()), (ax.yaxis, ax.get_yscale())]:
            if scale == "log":
                axis.set_major_locator(LogLocator(base=10, numticks=5))
                axis.set_minor_locator(NullLocator())
            else:
                axis.set_major_locator(MaxNLocator(nbins=5))
    fig.tight_layout()

LABEL = {
    "urms": r"$u_{\rm rms}$", "umax": r"$u_{\max}$", "brms": r"$B_{\rm rms}$",
    "erms": r"$E_{\rm rms}$", "emax": r"$E_{\max}$", "jrms": r"$J_{\rm rms}$ (magnetic)",
    "divErms": r"rms $\nabla\cdot E$", "J0rms": r"rms $J^0$", "rhoerms": r"rms $\rho_e$",
    "rhoem": r"$\langle\rho_e\rangle$", "gaussJ0rms": r"rms$(\nabla\cdot E-J^0)$",
    "gaussJ0max": r"max$|\nabla\cdot E-J^0|$", "gaussdamprms": r"rms $\gamma\sigma(\nabla\cdot E-J^0)$",
    "grms": r"rms$(\Gamma-\nabla\cdot A)$", "divJrms": r"rms $\nabla\cdot J$",
    "divJadvrms": r"rms$(u\cdot\nabla J^0+J^0\nabla\cdot u)$", "divJohmrms": r"rms $\sigma\gamma\nabla\cdot W$",
    "divJgamrms": r"rms $\sigma W\cdot\nabla\gamma$", "Johmrms": r"rms $|J|$", "J0urms": r"rms $|J^0u|$",
    "Jsigrms": r"rms $|\gamma\sigma W|$", "uEm": r"$\langle u\cdot E\rangle$", "uErms": r"rms $u\cdot E$",
    "f0m": r"$\langle E\cdot J\rangle$", "f0rms": r"rms $E\cdot J$", "fLorrms": r"rms $|J^0E+J\times B|$",
    "fJ0Erms": r"rms $|J^0E|$", "fJxBrms": r"rms $|J\times B|$", "gammam": r"$\langle\gamma\rangle$",
    "gammamax": r"$\max\gamma$", "BcurlEm": r"$\langle B\cdot\nabla\times E\rangle$", "ebm": r"$\langle E\cdot B\rangle$",
}
lab = lambda q: LABEL.get(q, q)

## Time series

In [ ]:
def load_ts(run):
    '''Read data/time_series.dat into a dict of arrays (header names from the '#' line).'''
    lines = (run / "data" / "time_series.dat").read_text().splitlines()
    header = [l for l in lines if l.startswith("#")][-1]
    names = [n for n in re.split(r"-+", header.lstrip("#")) if n]
    rows = [l.split() for l in lines if l.strip() and not l.startswith("#")]
    data = np.array(rows, dtype=float)
    return {n: data[:, i] for i, n in enumerate(names)}

TS = {N: load_ts(RUNS[N]) for N in NS}
NAMES = list(TS[NS[0]].keys())
print(len(NAMES), "diagnostics:", " ".join(NAMES))

## Checks
$t_N$ identical; finite; $r(0)=0$; $(\Gamma-\nabla\cdot\mathbf A)(0)=0$; $Q_N(0)$ equal for all $N$.

In [ ]:
t_ref = TS[NS[0]]["t"]
for N in NS:
    ts = TS[N]
    assert np.allclose(ts["t"], t_ref), f"N={N}: different output times"
    assert all(np.isfinite(v).all() for v in ts.values()), f"N={N}: NaN/inf in time series"
    assert ts["gaussJ0rms"][0] == 0.0 and ts["gaussJ0max"][0] == 0.0, f"N={N}: Gauss residual /= 0 at t=0"
    assert ts["grms"][0] == 0.0, f"N={N}: Gamma - div A /= 0 at t=0"
print("output times identical, no NaNs, Gauss residual = 0 at t=0 for all N   -> OK")

print("\nt=0 values (should agree up to discretization error):")
for q in ["urms", "brms", "erms", "divErms", "J0rms", "jrms"]:
    vals = [TS[N][q][0] for N in NS]
    spread = (max(vals) - min(vals)) / max(abs(np.mean(vals)), 1e-300)
    print(f"  {q:8s}", "  ".join(f"{v:.6e}" for v in vals), f"  rel. spread {spread:.1e}")

## All diagnostics

In [ ]:
skip = {"it", "t", "dt"}
qs = [n for n in NAMES if n not in skip]
ncol = 3
nrow = int(np.ceil(len(qs) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(7 * ncol, 4.6 * nrow), sharex=True)
for ax, q in zip(axes.flat, qs):
    for N in NS:
        ax.plot(TS[N]["t"], TS[N][q], color=COL[N], label=f"N={N}")
    if np.all(TS[NS[-1]][q][1:] > 0) and TS[NS[-1]][q][1:].max() / TS[NS[-1]][q][1:].min() > 20:
        ax.set_yscale("log")
    ax.set_title(f"{lab(q)}   [{q}]")
for ax in axes.flat[len(qs):]:
    ax.axis("off")
axes.flat[0].legend()
for ax in axes[-1]:
    ax.set_xlabel("t")
style(fig)

## Gauss residual
$r$ (`gaussJ0rms`, `gaussJ0max`), $\;r/\nabla\cdot\mathbf E$, $\;\gamma\sigma r$ (`gaussdamprms`), $\;\Gamma-\nabla\cdot\mathbf A$ (`grms`).

In [ ]:
panels = [("gaussJ0rms", lab("gaussJ0rms")),
          ("gaussJ0max", lab("gaussJ0max")),
          ("gauss_rel", r"rms$(\nabla\cdot E-J^0)$ / rms $\nabla\cdot E$"),
          ("gaussdamprms", lab("gaussdamprms")),
          ("grms", lab("grms"))]
fig, axes = plt.subplots(2, 3, figsize=(22, 12.5))
for ax, (q, title) in zip(axes.flat, panels):
    for N in NS:
        ts = TS[N]
        y = ts["gaussJ0rms"] / ts["divErms"] if q == "gauss_rel" else ts[q]
        ax.semilogy(ts["t"][1:], y[1:], color=COL[N], label=f"N={N}")
    ax.set_title(title); ax.set_xlabel("t")
axes.flat[0].legend()
axes.flat[-1].axis("off")
style(fig)

## Convergence
$r_N\propto N^{-p}$, $\quad p=\log_2(r_N/r_{2N})$; 6th-order derivatives $\Rightarrow p\to6$.

In [ ]:
t_check = [0.25, 0.5, 0.75, t_ref[-1]]
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for q, ax in zip(["gaussJ0rms", "grms"], axes):
    print(f"\n{q}: observed order p = log2(r_N / r_2N)")
    print("   t     " + "  ".join(f"{N}->{2*N}" for N in NS[:-1]))
    for k, tc in enumerate(t_check):
        i = int(np.argmin(abs(t_ref - tc)))
        r = np.array([TS[N][q][i] for N in NS])
        p = np.log2(r[:-1] / r[1:])
        print(f"  {t_ref[i]:.3f}  " + "  ".join(f"{x:7.2f}" for x in p))
        ax.loglog(NS, r, "o-", color=plt.cm.plasma(0.15 + 0.22 * k), label=f"t={t_ref[i]:.2f}")
    Nn = np.array(NS, float)
    r0 = TS[NS[0]][q][int(np.argmin(abs(t_ref - 0.5)))]
    for pp, ls in [(2, ":"), (4, "--"), (6, "-.")]:
        ax.loglog(Nn, r0 * (Nn / Nn[0]) ** (-pp), color="0.3", ls=ls, lw=1.8, label=f"$N^{{-{pp}}}$")
    ax.set_xlabel("N"); ax.set_ylabel(lab(q)); ax.set_title(lab(q))
    ax.set_xticks(NS); ax.set_xticklabels([str(N) for N in NS])
    ax.legend(ncol=2)
style(fig)
for ax in axes:   # keep N=16..128 as the x ticks
    ax.set_xticks(NS); ax.set_xticklabels([str(N) for N in NS]); ax.xaxis.set_minor_locator(NullLocator())

**Test:** $r_N>r_{2N}$; $\;p\ge2$ between the two finest grids.

In [ ]:
P_MIN = 2.0
for tc in t_check[1:]:
    i = int(np.argmin(abs(t_ref - tc)))
    r = np.array([TS[N]["gaussJ0rms"][i] for N in NS])
    assert np.all(np.diff(r) < 0), f"t={t_ref[i]:.2f}: residual not decreasing with N: {r}"
    if len(NS) >= 2:
        p = np.log2(r[-2] / r[-1])
        assert p >= P_MIN, f"t={t_ref[i]:.2f}: order {p:.2f} < {P_MIN} between N={NS[-2]} and N={NS[-1]}"
print("Gauss residual decreases with N at all check times, order >=", P_MIN, " -> OK")

## Term budgets
Solid: $N_{\max}$; dashed: coarser $N$.

| group | equation | diagnostics |
|---|---|---|
| continuity | $\nabla\cdot\mathbf J=\text{adv}+\text{ohm}+\text{gam}$ | `divJ*rms`, `gaussdamprms` |
| current | $\mathbf J=J^0\mathbf u+\gamma\sigma\mathbf W$ | `Johmrms`, `J0urms`, `Jsigrms` |
| force | $f^i=J^0\mathbf E+\mathbf J\times\mathbf B$, $\;f^0=\mathbf E\cdot\mathbf J$ | `fLorrms`, `fJ0Erms`, `fJxBrms`, `f0rms` |
| relativistic | $\gamma-1$, $\mathbf u\cdot\mathbf E$ | `gammam`, `gammamax`, `uErms` |

In [ ]:
groups = {
    "continuity terms": ["divJrms", "divJadvrms", "divJohmrms", "divJgamrms", "gaussdamprms"],
    "current": ["Johmrms", "J0urms", "Jsigrms", "jrms"],
    "Lorentz force": ["fLorrms", "fJ0Erms", "fJxBrms", "f0rms"],
    "charge": ["J0rms", "rhoerms", "divErms"],
}
Nf = NS[-1]
fig, axes = plt.subplots(2, 3, figsize=(24, 13))
for ax, (title, qs_) in zip(axes.flat, groups.items()):
    for k, q in enumerate(qs_):
        c = plt.cm.tab10(k)
        ax.semilogy(TS[Nf]["t"][1:], TS[Nf][q][1:], color=c, label=lab(q))
        for N in NS[:-1]:
            ax.semilogy(TS[N]["t"][1:], TS[N][q][1:], color=c, ls="--", lw=1.4, alpha=0.8)
    ax.set_title(f"{title} (N={Nf})"); ax.set_xlabel("t"); ax.legend(fontsize=13)
ax = axes.flat[len(groups)]
ax.semilogy(TS[Nf]["t"], TS[Nf]["gammam"] - 1, label=r"$\langle\gamma\rangle-1$")
ax.semilogy(TS[Nf]["t"], TS[Nf]["gammamax"] - 1, label=r"$\max\gamma-1$")
ax.semilogy(TS[Nf]["t"], TS[Nf]["uErms"], label=lab("uErms"))
ax.set_title(f"relativistic (N={Nf})"); ax.set_xlabel("t"); ax.legend()
axes.flat[-1].axis("off")
style(fig)

**Test:** $\mathrm{rms}(\nabla\cdot\mathbf J)\le\mathrm{rms}(\text{adv})+\mathrm{rms}(\text{ohm})+\mathrm{rms}(\text{gam})$.

In [ ]:
for N in NS:
    ts = TS[N]
    parts = ts["divJadvrms"] + ts["divJohmrms"] + ts["divJgamrms"]
    assert np.all(ts["divJrms"] <= parts * (1 + 1e-10)), f"N={N}: divJrms > sum of parts"
print("continuity budget consistent for all N (divJrms <= sum of parts) -> OK")
ts = TS[NS[-1]]
print(f"\nN={NS[-1]}, time-averaged term / divJrms:")
for q in ["divJadvrms", "divJohmrms", "divJgamrms", "gaussdamprms"]:
    print(f"  {q:13s} {np.mean(ts[q][1:] / ts['divJrms'][1:]):.3e}")

## Self-convergence
$|Q_N-Q_{N_{\max}}|/|Q_{N_{\max}}|$ at $t_{\rm end}$ (`print.in` here: 6 digits).

In [ ]:
quantities = ["urms", "brms", "erms", "J0rms", "jrms", "fLorrms"]
i = -1
Nf = NS[-1]
fig, ax = plt.subplots(figsize=(10, 7))
print(f"|Q_N - Q_{Nf}| / |Q_{Nf}| at t={t_ref[i]:.3f}")
for k, q in enumerate(quantities):
    d = np.array([abs(TS[N][q][i] - TS[Nf][q][i]) / abs(TS[Nf][q][i]) for N in NS[:-1]])
    print(f"  {q:8s}", "  ".join(f"{x:.2e}" for x in d))
    m = d > 0
    ax.loglog(np.array(NS[:-1])[m], d[m], "o-", color=plt.cm.tab10(k), label=lab(q))
Nn = np.array(NS[:-1], float)
ax.loglog(Nn, 1e-2 * (Nn / Nn[0]) ** -6, color="0.3", ls="-.", lw=1.8, label="$N^{-6}$")
ax.set_xlabel("N"); ax.set_ylabel(f"relative difference to N={Nf}"); ax.legend(ncol=2)
style(fig)
ax.set_xticks(NS[:-1]); ax.set_xticklabels([str(N) for N in NS[:-1]]); ax.xaxis.set_minor_locator(NullLocator())

## Snapshots
| | definition |
|---|---|
| $r_{\rm FD}$ | $D_6\cdot\mathbf E-J^0$, $\;D_6f=\dfrac{45(f_{+1}-f_{-1})-9(f_{+2}-f_{-2})+(f_{+3}-f_{-3})}{60\,\delta x}$ (as in code) |
| $r_{\rm spec}$ | $\mathcal F^{-1}[\,i\mathbf k\cdot\hat{\mathbf E}\,]-J^0$ (exact $\nabla\cdot\mathbf E$) |
| fields | `pc.read.var`: `ex, ey, ez`, `rhoe` $=J^0$; `VAR1…` (`VAR0` lacks the `divE` aux; the `divE` aux is from an RK substep, not used) |

In [ ]:
L = 2 * np.pi

def wavenumbers(shape):
    '''Angular wavenumbers on an (nz, ny, nx) grid of a periodic box of side L.'''
    nz, ny, nx = shape
    kx = 2 * np.pi * np.fft.fftfreq(nx, d=L / nx)
    ky = 2 * np.pi * np.fft.fftfreq(ny, d=L / ny)
    kz = 2 * np.pi * np.fft.fftfreq(nz, d=L / nz)
    return np.meshgrid(kz, ky, kx, indexing="ij")      # KZ, KY, KX

def fd6_div(vx, vy, vz):
    '''Divergence with Pencil's 6th-order centred first derivative (periodic), arrays in (z, y, x) order.'''
    def d(f, axis):
        h = L / f.shape[axis]
        r = lambda k: np.roll(f, -k, axis=axis)
        return (45 * (r(1) - r(-1)) - 9 * (r(2) - r(-2)) + (r(3) - r(-3))) / (60 * h)
    return d(vx, 2) + d(vy, 1) + d(vz, 0)

def read_snapshots(run):
    '''All snapshots VAR1, VAR2, ... of a run (VAR0 lacks the divE auxiliary and cannot be read).'''
    nsnap = len(list((run / "data" / "proc0").glob("VAR[0-9]*")))
    out = []
    for k in range(1, nsnap):
        v = pc.read.var(datadir=str(run / "data"), ivar=k, trimall=True, quiet=True)
        out.append({"t": float(v.t), "ex": v.ex, "ey": v.ey, "ez": v.ez, "J0": v.rhoe})
    return out

def residuals_hat(s):
    '''Fourier transforms of the Gauss residual r = div E - J0, with spectral and with FD6 divergence.'''
    KZ, KY, KX = wavenumbers(s["J0"].shape)
    J0h = np.fft.fftn(s["J0"])
    r_spec = 1j * (KX * np.fft.fftn(s["ex"]) + KY * np.fft.fftn(s["ey"]) + KZ * np.fft.fftn(s["ez"])) - J0h
    r_fd = np.fft.fftn(fd6_div(s["ex"], s["ey"], s["ez"])) - J0h
    return r_spec, r_fd

def rms_from_hat(fh):
    '''rms of the real-space field, via Parseval (numpy's fftn is unnormalized).'''
    return np.sqrt(np.sum(abs(fh) ** 2)) / fh.size

SNAP = {N: read_snapshots(RUNS[N]) for N in NS}
snap = {}
for N in NS:
    snap[N] = []
    for s in SNAP[N]:
        r_spec, r_fd = residuals_hat(s)
        snap[N].append({"t": s["t"], "fd": rms_from_hat(r_fd), "spec": rms_from_hat(r_spec),
                        "fd_vs_spec": rms_from_hat(r_fd - r_spec)})
for N in NS:
    print(f"N={N:3d}  " + "  ".join(f"t={s['t']:.2f}: fd {s['fd']:.2e} spec {s['spec']:.2e} |fd-spec| {s['fd_vs_spec']:.2e}"
                                for s in snap[N]))

**Test:** $\mathrm{rms}\,r_{\rm FD}=$ `gaussJ0rms` at equal $t$; $\;p_{\rm spec}\ge4$.

In [ ]:
for N in NS:
    # snapshots are written every dsnap, diagnostics every it1 steps; compare where the times coincide
    matched = 0
    for s in snap[N]:
        i = int(np.argmin(abs(TS[N]["t"] - s["t"])))
        if abs(TS[N]["t"][i] - s["t"]) > 1e-6:
            continue
        matched += 1
        assert np.isclose(s["fd"], TS[N]["gaussJ0rms"][i], rtol=1e-3), \
            f"N={N} t={s['t']:.2f}: snapshot {s['fd']:.4e} vs time series {TS[N]['gaussJ0rms'][i]:.4e}"
    assert matched > 0, f"N={N}: no snapshot at a diagnostic output time"
print("FD residual from snapshots reproduces gaussJ0rms -> OK")
k_last = min(len(snap[N]) for N in NS) - 1
spec_last = np.array([snap[N][k_last]["spec"] for N in NS])
p = np.log2(spec_last[:-1] / spec_last[1:])
print("spectral residual at t=%.2f:" % snap[NS[0]][k_last]["t"], spec_last, " order", p)
assert np.all(p >= 4), f"spectral residual order too low: {p}"
print("spectral residual converges -> OK")

## Residual spectrum
$$E_r(k)=\frac{1}{N^6}\sum_{k-\frac12\le|\mathbf k|<k+\frac12}|\hat r(\mathbf k)|^2,\qquad \sum_kE_r(k)=\langle r^2\rangle$$
Dotted: $k=N/2$.

In [ ]:
def shell_spectrum(fh):
    '''Shell-summed |fh|^2 / Ntot^2 in unit-width shells around integer k (box of side 2pi -> k in units of 1).'''
    KZ, KY, KX = wavenumbers(fh.shape)
    kshell = np.rint(np.sqrt(KX ** 2 + KY ** 2 + KZ ** 2) * L / (2 * np.pi)).astype(int)
    E = np.bincount(kshell.ravel(), weights=(abs(fh) ** 2).ravel()) / fh.size ** 2
    return np.arange(E.size), E

SPEC = {}   # SPEC[N][isnap] = dict(t, k, spec, fd)
for N in NS:
    SPEC[N] = []
    for s in SNAP[N]:
        r_spec, r_fd = residuals_hat(s)
        k, Es = shell_spectrum(r_spec)
        _, Ef = shell_spectrum(r_fd)
        SPEC[N].append({"t": s["t"], "k": k, "spec": Es, "fd": Ef})

def plot_spectra(ax, which, isnap, title):
    for N in NS:
        d = SPEC[N][isnap]
        m = (d["k"] >= 1) & (d[which] > 0)
        ax.loglog(d["k"][m], d[which][m], color=COL[N], label=f"N={N}")
        ax.axvline(N / 2, color=COL[N], ls=":", lw=1.6)
    ax.set_xlabel("k"); ax.set_ylabel(r"$E_r(k)$"); ax.set_title(title)

isnap = min(len(SPEC[N]) for N in NS) - 1
t_s = SPEC[NS[0]][isnap]["t"]
fig, axes = plt.subplots(1, 2, figsize=(20, 7.5), sharey=True)
plot_spectra(axes[0], "spec", isnap, rf"spectral $\nabla\cdot E$,  t={t_s:.2f}")
plot_spectra(axes[1], "fd", isnap, rf"FD6 $\nabla\cdot E$ (as in code),  t={t_s:.2f}")
axes[0].legend()
style(fig)

$E_r(k,t)$ for each $N$ ($r_{\rm spec}$).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 14), sharey=True)
for ax, N in zip(axes.flat, NS):
    for j, d in enumerate(SPEC[N]):
        m = (d["k"] >= 1) & (d["spec"] > 0)
        ax.loglog(d["k"][m], d["spec"][m], color=plt.cm.plasma(0.15 + 0.6 * j / max(len(SPEC[N]) - 1, 1)),
                  label=f"t={d['t']:.2f}")
    ax.axvline(N / 2, color="0.4", ls=":", lw=1.6)
    ax.set_title(f"N={N}"); ax.set_xlabel("k"); ax.set_ylabel(r"$E_r(k)$"); ax.legend()
for ax in axes.flat[len(NS):]:
    ax.axis("off")
style(fig)

**Test:** $\sum_kE_r(k)=\langle r^2\rangle$ for $r_{\rm FD}$ and $r_{\rm spec}$.

In [ ]:
for N in NS:
    for d, s in zip(SPEC[N], snap[N]):
        assert np.isclose(d["spec"].sum(), s["spec"] ** 2, rtol=1e-10), f"N={N} t={d['t']:.2f}: Parseval (spectral)"
        assert np.isclose(d["fd"].sum(), s["fd"] ** 2, rtol=1e-10), f"N={N} t={d['t']:.2f}: Parseval (FD6)"
print("sum_k E_r(k) = <r^2> for all runs and snapshots -> OK\n")
print(f"fraction of residual power at k <= 4 (t={t_s:.2f}):")
for N in NS:
    d = SPEC[N][isnap]
    print(f"  N={N:3d}  spectral {d['spec'][:5].sum() / d['spec'].sum():.3f}   FD6 {d['fd'][:5].sum() / d['fd'].sum():.3f}")